In [1]:
! pip install brian2 -q

In [2]:
# Store outputs beside the notebook without cloning or linking to another notebook.
from pathlib import Path
OUTPUT_DIR = Path.cwd() / 'G5.5_results'
FIGURES_DIR = OUTPUT_DIR / 'figures'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
CONDITION = 'G5.5'


In [3]:
from brian2 import *
import numpy as np
import matplotlib.pyplot as plt

from scipy.signal import welch
from scipy.integrate import quad
from scipy.special import erfc
from scipy.optimize import brentq
from scipy.fft import fft
from scipy.ndimage import gaussian_filter1d

import warnings

warnings.filterwarnings('ignore')
prefs.codegen.target = 'cython'

In [4]:
# Standalone recurrent-network setup. No repository or cache is required.
start_scope()
defaultclock.dt = 0.1 * ms
seed(4321)

N_E = 100000
gamma = 0.25
N_I = int(gamma * N_E)
N = N_E + N_I

epsilon = 0.01
C_E = int(epsilon * N_E)
C_I = int(epsilon * N_I)

tau = 20 * ms
theta = 20 * mV
V_r = 10.0 * mV
tau_rp = 2 * ms

J = 1.0 * mV
g = 5.5
D = 1.5 * ms
RI = 30 * mV
sim_time = 3000 * ms

neurons = NeuronGroup(
    N, name='sc_neurons',
    model='dv/dt = (-v + RI) / tau : volt (unless refractory)',
    threshold='v > theta',
    reset='v = V_r',
    refractory=tau_rp,
    method='euler',
)
neurons.v = 'rand() * mV'
exc_neurons = neurons[:N_E]
inh_neurons = neurons[N_E:]


In [5]:
# Recurrent synapses with the fixed in-degree used in the original model.
exc_syn = Synapses(exc_neurons, neurons, name="sc_exc", on_pre="v += J", delay=D)
inh_syn = Synapses(inh_neurons, neurons, name="sc_inh", on_pre="v += -g*J", delay=D)

targets_E = np.repeat(np.arange(N), C_E)
targets_I = np.repeat(np.arange(N), C_I)
sources_E = np.random.randint(0, N_E, size=N * C_E)
sources_I = np.random.randint(0, N_I, size=N * C_I)
exc_syn.connect(i=sources_E, j=targets_E)
inh_syn.connect(i=sources_I, j=targets_I)


In [6]:
# Run directly and checkpoint every simulated second.
from pathlib import Path
checkpoint_dir = OUTPUT_DIR / 'checkpoints'
checkpoint_dir.mkdir(parents=True, exist_ok=True)

rate_mon = PopulationRateMonitor(neurons)
spike_mon = SpikeMonitor(neurons)
net = Network(neurons, exc_syn, inh_syn, rate_mon, spike_mon)

checkpoint_files = sorted(checkpoint_dir.glob('network-*.brian'))
start_s = 0.0
if checkpoint_files:
    latest = checkpoint_files[-1]
    net.restore('checkpoint', filename=str(latest), restore_random_state=True)
    start_s = float(net.t / second)
    print(f'Resuming recurrent network at {start_s:g} s')

target_s = float(sim_time / second)
while start_s < target_s - 1e-12:
    amount_s = min(1.0, target_s - start_s)
    net.run(amount_s * second)
    start_s = float(net.t / second)

    network_file = checkpoint_dir / f'network-{start_s:07.3f}s.brian'
    net.store('checkpoint', filename=str(network_file))
    np.savez_compressed(
        checkpoint_dir / f'monitor-{start_s:07.3f}s.npz',
        spike_indices=np.asarray(spike_mon.i),
        spike_times_ms=np.asarray(spike_mon.t / ms),
        population_rate_time_ms=np.asarray(rate_mon.t / ms),
        population_rate_hz=np.asarray(rate_mon.rate / Hz),
    )
    print(f'Checkpoint: recurrent {start_s:g}/{target_s:g} s')

# Final monitor checkpoint always marks a completed run.
np.savez_compressed(
    OUTPUT_DIR / 'G5.5_completed_monitor.npz',
    spike_indices=np.asarray(spike_mon.i),
    spike_times_ms=np.asarray(spike_mon.t / ms),
    population_rate_time_ms=np.asarray(rate_mon.t / ms),
    population_rate_hz=np.asarray(rate_mon.rate / Hz),
)


Checkpoint: recurrent 1/3 s
Checkpoint: recurrent 2/3 s
Checkpoint: recurrent 3/3 s


In [7]:
# Extract spike trains from the monitor
spike_trains = spike_mon.spike_trains()
cv_list = []

# Iterate through each neuron's spike train
for neuron_idx, train in spike_trains.items():

    # Ensure there are enough spikes to calculate an ISI
    if len(train) > 2:

        # Calculate Inter-Spike Intervals (ISIs)
        isi = np.diff(train / ms)

        # Calculate CV: Standard Deviation / Mean
        cv = np.std(isi) / np.mean(isi)
        cv_list.append(cv)

# Calculate the network-wide average CV
if cv_list:
    mean_cv = np.mean(cv_list)
    print(f"Average Network CV: {mean_cv:.3f}")
else:
    print("Not enough spikes to calculate CV.")


average_firing_rate = np.mean(rate_mon.rate)
print(f"Average Firing Rate: {average_firing_rate:.2f} Hz")

Average Network CV: 2.530
Average Firing Rate: 16.53 Hz
